In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import shutil
import seaborn as sns
import glob
from pathlib import Path
from tqdm import tqdm

import cv2
from PIL import Image
import albumentations as A
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as F
from torch.utils.data import DataLoader , Dataset , random_split
from torchmetrics.regression import MeanAbsoluteError
import lightning as L


from ultralytics import YOLO
import timm 
from transformers import ViTModel, ViTConfig
from autogluon.tabular import TabularDataset, TabularPredictor


/lustrefs/disk/project/ai901504-ai0004/501641_Big/week5/env_image/lib/python3.10/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno 101] Network is unreachable>
  data = fetch_version_info()


In [ ]:
## Vit Augment / preprocess (Gamma, intensity) / class balance Pure model
# Train Loss: tensor(3.2919) / Validation Loss: tensor(7.3910)

#################################################################
## Vit-Built Augment / preprocess (Gamma, intensity)
# Train Loss: tensor(3.4057) / Validation Loss: tensor(14.7791)

## balance classes
# Final Train Loss: tensor(4.9684) / Validation Loss: tensor(15.7831)

##################################################################
## YOLO-Built Augment / preprocess (Gamma, intensity) YOLO11x-cls
# Final Train Loss: tensor(15.5159) / Validation Loss: tensor(15.8344)

## balance classes
# Final Train Loss: tensor(6.2480) / Validation Loss: tensor(24.6569)

## Conv2D / balance classes
# Final Train Loss: tensor(0.4125) / Validation Loss: tensor(4.9564)

## Con2D / Normal set
# Final Train Loss: tensor(0.2309) / Validation Loss: tensor(10.4011)

###################################################################
## YOLO-Built Augment / preprocess (Gamma, intensity) YOLO12x-cls
# Final Train Loss: tensor(0.9750) / Validation Loss: tensor(6.3337)

## balance classes
# Final Train Loss: tensor(1.4072) / Validation Loss: tensor(7.4845)

####################################################################
## YOLO-Pertrain and fine-tune # Pretrain - loss 0.7475 - acc 0.791 on normal 
## YOLO11x-cls / Augment / preprocess / balance
# Final Train Loss: tensor(1.6344) / Validation Loss: tensor(18.6605)

# Normal set / balance
# Final Train Loss: tensor(0.9199) / Validation Loss: tensor(14.5770)

####################################################################
## Autogluon Tabular
# 

# Model

### Regressor (focus) v.1

In [ ]:
train_path = "/project/ai901504-ai0004/501641_Big/week5/train_augment.csv"
# train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/501641_Big/week5/dataset_2k"

In [ ]:
train_df = pd.read_csv(train_path)
train_df.head()

In [ ]:
# Separate the F0 class and the rest
f0 = train_df[train_df['TE result'] == 'F0']
other_classes = train_df[train_df['TE result'] != 'F0']

# Downsample F0 to 400 samples (random_state ensures reproducibility)
f0_downsampled = f0.sample(n=400, random_state=42)

# Combine back
train_df = pd.concat([f0_downsampled, other_classes], ignore_index=True)

# Optional: shuffle the final dataframe
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
class_list = train_df["TE result"].unique().tolist()
print(class_list)

class CustomDataSet(Dataset):
    def __init__(self, csv_file, image_dir, class_list, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        filename = self.df.image_name[index]
        if not filename.endswith(".png"):
            filename += ".png"
        image_path = os.path.join(self.image_dir, filename)
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label,dtype = torch.float32)  # Ensure label is a tensor

In [ ]:
BATCH_SIZE = 256
IMAGE_SIZE = 224
MEAN = [0.5, 0.5, 0.5]
STD = [0.5, 0.5, 0.5]

transformation = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN, std=STD),
    ])

dataset = CustomDataSet(train_path, image_dir,class_list, transform=transformation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
class LiverFibrosisModel(L.LightningModule):
    def __init__(self, num_classes=1):
        super(LiverFibrosisModel, self).__init__()

        # Load pretrained ViT base model
        self.vit = ViTModel.from_pretrained('/project/ai901504-ai0004/VisionModels/project/ai901504-ai0004/VisionModels/models--google--vit-large-patch16-224')

        # Freeze the ViT encoder
        for param in self.vit.parameters():
            param.requires_grad = True

        self.regressor = nn.Linear(self.vit.config.hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        cls_token = outputs.last_hidden_state[:, 0]  # CLS token
        regression_output = self.regressor(cls_token)
        return self.relu(regression_output)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        val_loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)


In [ ]:
model = LiverFibrosisModel()

trainer = L.Trainer(max_epochs=20,
                    log_every_n_steps = 10)
trainer.fit(model=model,
            train_dataloaders=train_loader,
            val_dataloaders = eval_loader
            )

In [ ]:
print("Final Train Loss:", trainer.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer.callback_metrics.get("val_loss"))

#### inference

In [ ]:
class_list = train_df["TE result"].unique().tolist()
print(class_list)

class CustomDataSetTest(Dataset):
    def __init__(self, csv_file, image_dir,class_list, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)


        if self.transform:
            image = self.transform(image)
        return image

In [ ]:
test_dataset = CustomDataSetTest(test_path, image_dir,class_list, transform=transformation)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
predictions = trainer.predict(model, test_loader)

In [ ]:
all_pred = torch.cat(predictions,dim = 0)
pred_array = all_pred.detach().numpy()
len(pred_array)

In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv")
test_df.head()

In [ ]:
test_df["TE(kPa)"] = pred_array
test_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=test_df, x="TE(kPa)", bins=30, kde=False, color='skyblue')

plt.title("CNN Answer Distribution")
plt.xlabel("TE(kPa)")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
test_df['TE(kPa)'] += 0.5

In [ ]:
test_df.loc[test_df['TE(kPa)'] > 11, 'TE(kPa)'] = 11

In [ ]:
test_df.to_csv("submissionViT.csv",index = False)

### YOLO

In [ ]:
model = YOLO("yolo11x-cls.yaml")

In [ ]:
!nvcc --version

In [ ]:
import torch
print(torch.__version__)
print(torch.version.cuda)


In [ ]:
path_dataset = '/project/ai901504-ai0004/501641_Big/week5/dataset_te_yolo_preprocess/'

device = 'cuda' # or 'cpu'
model.to(device)

results = model.train(
    data = path_dataset,
    epochs=100,
    imgsz=64
    )

### Focus model builder

In [ ]:
ViT = ViTModel.from_pretrained('/project/ai901504-ai0004/VisionModels/project/ai901504-ai0004/VisionModels/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50')

In [ ]:
train_path = "/project/ai901504-ai0004/501641_Big/week5/train_augment.csv"
# train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/501641_Big/week5/dataset_2k"

In [ ]:
train_df = pd.read_csv(train_path)
print(train_df)

In [ ]:
# Separate the F0 class and the rest
f0 = train_df[train_df['TE result'] == 'F0']
other_classes = train_df[train_df['TE result'] != 'F0']

# Downsample F0 to 400 samples (random_state ensures reproducibility)
f0_downsampled = f0.sample(n=400, random_state=42)

# Combine back
train_df = pd.concat([f0_downsampled, other_classes], ignore_index=True)

# Optional: shuffle the final dataframe
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
class CustomDataSet(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label,dtype = torch.float32)  # Ensure label is a tensor

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 224
MEAN = [0.485,0.456,0.406]
STD = [0.229,0.224,0.225]

transformation = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])
dataset = CustomDataSet(train_df, image_dir, transform=transformation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)
eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
image , label = next(iter(train_loader))
print(image.shape, label.shape,label[0])

In [ ]:
image , label = next(iter(eval_loader))
print(image.shape, label.shape)

In [ ]:
class LiverFibrosisModel(L.LightningModule):
    def __init__(self, model,num_classes=1):
        super(LiverFibrosisModel, self).__init__()

        # Load pretrained ViT base model
        self.vit = model
        # Freeze the ViT encoder
        for param in self.vit.parameters():
            param.requires_grad = False

        for param in self.vit.encoder.layer[-5:].parameters():
            param.requires_grad = True

        self.regressor = nn.Linear(self.vit.config.hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        cls_token = outputs.last_hidden_state[:, 0]  # CLS token
        regression_output = self.regressor(cls_token)
        return self.relu(regression_output)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        val_loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-5)


In [ ]:
model = LiverFibrosisModel(ViT)
model

In [ ]:
trainer = L.Trainer(max_epochs=30,
                    log_every_n_steps = 10)
trainer.fit(model=model,
            train_dataloaders=train_loader,
            val_dataloaders = eval_loader
            )

In [ ]:
print("Final Train Loss:", trainer.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer.callback_metrics.get("val_loss"))

### Yolo built 

In [ ]:
YOLO_model = YOLO("yolo11x-cls.yaml")
my_model = YOLO_model.model.model

In [ ]:
train_path = "/project/ai901504-ai0004/501641_Big/week5/train_augment.csv"
# train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/501641_Big/week5/dataset_2k"

In [ ]:
train_df = pd.read_csv(train_path)
print(train_df)

In [ ]:
# Separate the F0 class and the rest
f0 = train_df[train_df['TE result'] == 'F0']
other_classes = train_df[train_df['TE result'] != 'F0']

# Downsample F0 to 400 samples (random_state ensures reproducibility)
f0_downsampled = f0.sample(n=400, random_state=42)

# Combine back
train_df = pd.concat([f0_downsampled, other_classes], ignore_index=True)

# Optional: shuffle the final dataframe
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
class CustomDataSet(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label,dtype = torch.float32)  # Ensure label is a tensor

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 224
MEAN = [0.5,0.5,0.5]
STD = [0.5,0.5,0.5]

transformation = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])
dataset = CustomDataSet(train_df, image_dir, transform=transformation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
print(my_model)

In [ ]:
# my_model[9] = nn.Sequential(
#     nn.Conv2d(768, 256, kernel_size=1),
#     nn.ReLU(),
#     nn.AdaptiveAvgPool2d(1),
#     nn.Flatten(),
#     nn.Dropout(p=0.5),  # Dropout for MC Dropout
#     nn.Linear(256, 1)
# )

my_model[10] = nn.Sequential(
    nn.Conv2d(768, 1280, kernel_size=1),
    nn.BatchNorm2d(1280, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(p=0.5),
    nn.Linear(1280, 1)
)

# my_model[10] = nn.Sequential( nn.Softmax(), nn.Flatten(), nn.Linear(37632, 1))

In [ ]:
class LiverFeatureExtract(L.LightningModule):
    def __init__(self, model, mc_iteration):
        super(LiverFeatureExtract, self).__init__()
        self.model = model
        self.metric = MeanAbsoluteError()
        self.dropout = nn.Dropout()
        self.mc_iteration = mc_iteration

        for param in self.model.parameters():
            param.requires_grad = True
        
        # for param in self.vit.encoder.layer[-2:].parameters():
        #     param.requires_grad = True

    def forward(self, x):
        outputs = self.model(x)
        return outputs

    def predict(self, batch, mc=False):
        images, labels = batch

        if mc == True:
            pred = [self(images).unsqueeze(0) for _ in range(self.mc_iteration)]
            logits = torch.vstack(pred).mean(dim=0).squeeze()
            return logits, labels

        logits = self(images).squeeze()
        return logits ,labels

    def cal_loss(self, logits, labels):
        loss = F.mse_loss(logits, labels)
        metric = self.metric( logits,labels)
        return loss, metric

    def training_step(self, batch):
        logits, labels = self.predict(batch)
        loss, metric = self.cal_loss(logits, labels)
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        loss, metric = self.cal_loss(logits_clamped, labels)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def predict_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        return logits_clamped, labels

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

In [ ]:
model = LiverFeatureExtract(my_model, 10)

In [ ]:
trainer = L.Trainer(
    # accelerator="gpu",
    # devices=3,
    # strategy="ddp_notebook",  # for multi-GPU
    max_epochs=100,
    log_every_n_steps=10
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=eval_loader
)

In [ ]:
print("Final Train Loss:", trainer.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer.callback_metrics.get("val_loss"))

#### Inference

In [ ]:
class CustomDataSetTest(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]  # ✅ include label

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)
    
test_dataset = CustomDataSetTest(test_path, image_dir, transform=transformation)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
predictions = trainer.predict(model, test_loader)

In [ ]:
# Extract only prediction tensors (assuming first item in tuple)
pred_tensors = [batch[0] for batch in predictions]  # extract y_hat

# Concatenate into a single tensor
all_pred = torch.cat(pred_tensors, dim=0)

# Convert to NumPy
pred_array = all_pred.detach().cpu().numpy()

# Check the number of predictions
print(len(pred_array))


In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv")
test_df["TE(kPa)"] = pred_array
test_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=test_df, x="TE(kPa)", bins=30, kde=False, color='skyblue')

plt.title("CNN Answer Distribution")
plt.xlabel("TE(kPa)")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
test_df.to_csv("submissionCon2D.csv",index = False)

# Full-process deep-neural-network

In [ ]:
YOLO_model = YOLO("yolo11x-cls.yaml")

### Preprocess function

In [ ]:
def apply_preprocessing(folder_path, preprocessing_function, file_type):
    input_paths = list(Path(folder_path).glob(f"*.{file_type}"))
    for path in tqdm(input_paths, desc=f"Processing {folder_path}"):
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        img = preprocessing_function(img)
        cv2.imwrite(str(path), img)

In [ ]:
def histogram_equalization(image):
    # Apply Histogram Equalization
    equalized_image = cv2.equalizeHist(image)
    
    return equalized_image

In [ ]:
def gamma_correction(image, gamma=0.6):
    
    inv_gamma = 1.0 / gamma
    
    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255 for i in np.arange(256)
    ]).astype("uint8")
    
    return cv2.LUT(image, table)

In [ ]:
def intensity_transformation(image, alpha=1.5, beta=10):
  new_image = cv2.convertScaleAbs(image, alpha=alpha, beta=beta)
  return new_image

In [ ]:
def distillation_transformation(original_image, 
                                apply_denoising=True, 
                                clahe_clip_limit=2.0, 
                                clahe_tile_grid_size=(8, 8), 
                                median_blur_ksize=5):

    clahe = cv2.createCLAHE(clipLimit=clahe_clip_limit, tileGridSize=clahe_tile_grid_size)
    clahe_enhanced_image = clahe.apply(original_image)


    if apply_denoising:
        distilled_image = cv2.medianBlur(clahe_enhanced_image, median_blur_ksize)
        title = f"Distilled Image (CLAHE + Median Blur)"
    else:
        distilled_image = clahe_enhanced_image
        title = f"Distilled Image (CLAHE Only)"

    return distilled_image

In [ ]:
def extract_roi_from_sonogram(image):
    threshold_value, thresheld_sample_image = cv2.threshold(
        image,
        thresh = image.min(), # Use minimum grayscale value to separate the foreground from the background
        maxval = 255,
        # type = cv2.THRESH_BINARY_INV
        type = cv2.THRESH_BINARY_INV + cv2.THRESH_TRIANGLE
    )

    # Define a kernel for morphological operations
    kernel = np.ones((3, 3), np.uint8)

    # This helps fill small holes within objects
    closed_thresheld_sample_image = cv2.morphologyEx(
        thresheld_sample_image,
        cv2.MORPH_CLOSE,
        kernel
    )

    # Apply median filter to the closed thresholded image
    median_filtered_closed_thresheld_sample_image = cv2.medianBlur(
        closed_thresheld_sample_image,
        25
    ) # Kernel size is 25x25

    padded_median_filtered_closed_thresheld_sample_image = np.copy(median_filtered_closed_thresheld_sample_image)
    padded_median_filtered_closed_thresheld_sample_image[0, :] = 255
    padded_median_filtered_closed_thresheld_sample_image[-1, :] = 255
    padded_median_filtered_closed_thresheld_sample_image[:, 0] = 255
    padded_median_filtered_closed_thresheld_sample_image[:, -1] = 255

    edges_detected_sample_image = cv2.Canny(
        padded_median_filtered_closed_thresheld_sample_image,
        100, # I don't know what this parameter does, so let's leave it like this.
        200 # I don't know what this parameter does, so let's leave it like this.
    )

    kernel_close_edges = np.ones((31, 31), np.uint8) # Use a larger kernel for connecting broken edges
    closed_edges_detected_sample_image = cv2.morphologyEx(
        edges_detected_sample_image,
        cv2.MORPH_CLOSE,
        kernel_close_edges
    )

    # Find contours in the Canny edge image
    contours, hierarchy = cv2.findContours(
        closed_edges_detected_sample_image,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    # Create a copy of the original image to draw contours on
    contours_detected_image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)

    # Draw all contours on the image
    cv2.drawContours(contours_detected_image, contours, -1, (0, 255, 0), 2) # Green color, thickness 2

    # # Plot the image with all contours
    # plt.figure(figsize=(8, 8))
    # plt.imshow(contours_detected_image)
    # plt.title("Contours found in Canny edges")
    # plt.axis('off')
    # plt.show()

    # Find the largest contour
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)

        # Create a black image of the same size as the original image
        mask = np.zeros_like(image)

        # Draw the largest contour filled with white color on the mask
        cv2.drawContours(mask, [largest_contour], -1, (255), thickness=cv2.FILLED)

        # Extract the region inside the largest contour from the original image
        extracted_region = cv2.bitwise_and(image, image, mask=mask)

        # # Plot the extracted region
        # plt.figure(figsize=(8, 8))
        # plt.imshow(extracted_region, cmap='gray')
        # plt.title("Region inside the largest contour")
        # plt.axis('off')
        # plt.show()
    else:
        plt.figure(figsize=(6, 6))
        plt.imshow(image, cmap='gray')
        plt.title("No Contours Found")
        plt.axis('off')
        plt.show()
        return image
        # raise NotImplementedError("No contours found.")
    
    return extracted_region

In [ ]:
def pseudocolor_image(original_image, colormap_type=cv2.COLORMAP_JET):

    pseudocolored_image = cv2.applyColorMap(original_image, colormap_type)

    # แปลงชื่อ Colormap ให้เป็น String สำหรับใช้ใน Title
    colormap_name = "Custom Colormap"
    if colormap_type == cv2.COLORMAP_JET: colormap_name = "JET"
    elif colormap_type == cv2.COLORMAP_BONE: colormap_name = "BONE"
    elif colormap_type == cv2.COLORMAP_HOT: colormap_name = "HOT"
    elif colormap_type == cv2.COLORMAP_RAINBOW: colormap_name = "RAINBOW"
    elif colormap_type == cv2.COLORMAP_VIRIDIS: colormap_name = "VIRIDIS"
    elif colormap_type == cv2.COLORMAP_PARULA: colormap_name = "PARULA"
    elif colormap_type == cv2.COLORMAP_MAGMA: colormap_name = "MAGMA"
    elif colormap_type == cv2.COLORMAP_INFERNO: colormap_name = "INFERNO"
    elif colormap_type == cv2.COLORMAP_PLASMA: colormap_name = "PLASMA"


    title = f"Pseudocolored Image ({colormap_name} Colormap)"

    return pseudocolored_image

### Dataset for pretrain

In [ ]:
# rm -r /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain

# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val

# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train/F0
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train/F1
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train/F2
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train/F3n/train/F3
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train/F4

# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val/F0
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val/F1
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val/F2
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val/F3
# mkdir /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val/F4

# File path
# /project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/liver-lesion/annotations/51.txt
# /project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/liver-lesion/images/51.jpg

#Example txt file content
# 2 0.5638888888888889 0.4074074074074074 0.058333333333333334 0.08888888888888889
# 2 0.36041666666666666 0.48518518518518516 0.08194444444444444 0.09259259259259259

#Use rm -r /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain && mkdir -p /project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/{train,val}

In [ ]:
dataset_train = Path(f'/project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/train')
dataset_val = Path(f'/project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/val')
image_path = Path('/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/liver-lesion/images/')
annotation_path = Path('/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/liver-lesion/annotations/')

In [ ]:
image_list = list(annotation_path.glob('*.txt'))
train_list, eval_list = train_test_split(image_list, test_size=0.2, random_state=42)

In [ ]:
# histogram_equalization(image)
# gamma_correction(image)
# intensity_transformation(image)
# distillation_transformation(image)
# extract_roi_from_sonogram(image)
# pseudocolor_image(image)


def copy_files(file_list, target_root):
    for annotation_file in file_list:
        with open(annotation_file, 'r') as file:
            first_line = file.readline().strip()
            if not first_line:
                continue  # skip empty files
            i = int(first_line.split()[0])

            if i > 4:
                continue  # skip if i is not in [0, 4]

            # Define target folder
            target_folder = target_root / f'F{i}'
            target_folder.mkdir(parents=True, exist_ok=True)

            # Define image file path (same name but .jpg/.png assumed)
            base_name = annotation_file.stem
            source_image_file = image_path / f'{base_name}.jpg'

            image = cv2.imread(str(source_image_file), cv2.IMREAD_GRAYSCALE)
            # image = extract_roi_from_sonogram(image)
            # image = gamma_correction(image, gamma=0.6)
            # image = intensity_transformation(image)

            output_image_path = target_folder / source_image_file.name
            cv2.imwrite(str(output_image_path), image)

In [ ]:
copy_files(train_list, dataset_train)
copy_files(eval_list, dataset_val)

In [ ]:
for i in range(5):
    train_images = list((dataset_train / f'F{i}').glob('*.jpg'))
    val_images = list((dataset_val / f'F{i}').glob('*.jpg'))

    print(f"Class F{i}:")
    print(f"  Train images: {len(train_images)}")
    print(f"  Validation images: {len(val_images)}")

### YOLO classifier

In [ ]:
path_dataset = '/project/ai901504-ai0004/501641_Big/week5/dataset_for_pretrain/'

device = 'cuda' # or 'cpu'
YOLO_model.to(device)

results = YOLO_model.train(
    data = path_dataset,
    epochs=100,
    imgsz=224
    )

### fine-tune

In [ ]:
import copy
my_model = copy.deepcopy(YOLO_model.model.model)

In [ ]:
my_model

In [ ]:
train_path = "/project/ai901504-ai0004/501641_Big/week5/train_augment.csv"
# train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/501641_Big/week5/dataset_2k"

In [ ]:
train_df = pd.read_csv(train_path)
print(train_df)

In [ ]:
# Separate the F0 class and the rest
f0 = train_df[train_df['TE result'] == 'F0']
other_classes = train_df[train_df['TE result'] != 'F0']

# Downsample F0 to 400 samples (random_state ensures reproducibility)
f0_downsampled = f0.sample(n=400, random_state=42)

# Combine back
train_df = pd.concat([f0_downsampled, other_classes], ignore_index=True)

# Optional: shuffle the final dataframe
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
class CustomDataSet(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label,dtype = torch.float32)  # Ensure label is a tensor

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = 224
MEAN = [0.5,0.5,0.5]
STD = [0.5,0.5,0.5]

transformation = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])
dataset = CustomDataSet(train_df, image_dir, transform=transformation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
print(my_model[10])

In [ ]:
# my_model[9] = nn.Sequential(
#     nn.Conv2d(768, 256, kernel_size=1),
#     nn.ReLU(),
#     nn.AdaptiveAvgPool2d(1),
#     nn.Flatten(),
#     nn.Dropout(p=0.5),  # Dropout for MC Dropout
#     nn.Linear(256, 1)
# )

my_model[10] = nn.Sequential(
    nn.Conv2d(768, 1280, kernel_size=1),
    nn.BatchNorm2d(1280),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(p=0.5),
    nn.Linear(1280, 1)
)

# my_model[10] = nn.Sequential(
#     nn.Conv2d(768, 256, kernel_size=1),
#     nn.ReLU(),
#     nn.AdaptiveAvgPool2d(1),
#     nn.Flatten(),
#     nn.Dropout(p=0.5),
#     nn.Linear(256, 1)
# )

# my_model[10] = nn.Sequential( nn.Softmax(), nn.Flatten(), nn.Linear(37632, 1))

In [ ]:
# class LiverFeatureExtract(L.LightningModule):
#     def __init__(self, model, mc_iteration, learning_rate=1e-4, freeze_backbone=True):
#         super(LiverFeatureExtract, self).__init__()
#         self.model = model
#         self.metric = MeanAbsoluteError()
#         self.mc_iteration = mc_iteration
#         self.learning_rate = learning_rate
        
#         # Modify the classification head for regression
#         self._modify_head_for_regression()
        
#         # Freeze/unfreeze layers
#         if freeze_backbone:
#             self._freeze_backbone()
#         else:
#             self._unfreeze_all()

#     def _modify_head_for_regression(self):
#         """Modify the classification head for regression output"""
#         # Access the Classify module (layer 10)
#         classify_module = self.model[10]
        
#         # Keep the convolutional layers and pooling
#         # Only replace the final linear layer
#         in_features = classify_module.linear.in_features
        
#         # Replace the linear layer for regression (output size = 1)
#         classify_module.linear = nn.Linear(in_features, 1)
        
#         # Optionally adjust dropout
#         classify_module.drop = nn.Dropout(p=0.2)  # Reduced dropout for regression

#     def _freeze_backbone(self):
#         """Freeze all layers except the classification head"""
#         # Freeze layers 0-9 (backbone)
#         for i in range(10):
#             for param in self.model[i].parameters():
#                 param.requires_grad = False
        
#         # Unfreeze layer 10 (classification head)
#         for param in self.model[10].parameters():
#             param.requires_grad = True

#     def _unfreeze_all(self):
#         """Unfreeze all parameters"""
#         for param in self.model.parameters():
#             param.requires_grad = True

#     def _unfreeze_last_n_layers(self, n):
#         """Unfreeze last n layers for progressive fine-tuning"""
#         # Freeze all first
#         for param in self.model.parameters():
#             param.requires_grad = False
        
#         # Unfreeze last n layers
#         total_layers = len(self.model)
#         start_idx = max(0, total_layers - n)
        
#         for i in range(start_idx, total_layers):
#             for param in self.model[i].parameters():
#                 param.requires_grad = True

#     def forward(self, x):
#         outputs = self.model(x)
#         if isinstance(outputs, tuple):
#             outputs = outputs[0]  # Use only the main output
#         return outputs

#     def predict(self, batch, mc=False):
#         images, labels = batch

#         if mc:
#             self.train()  # Enable dropout
#             pred = [self(images).unsqueeze(0) for _ in range(self.mc_iteration)]
#             self.eval()  # Disable dropout
#             logits = torch.vstack(pred).mean(dim=0).squeeze()
#             return logits, labels

#         logits = self(images).squeeze()
#         return logits, labels

#     def cal_loss(self, logits, labels):
#         # Ensure same shape for loss calculation
#         logits = logits.view(-1)
#         labels = labels.view(-1).float()
        
#         loss = F.mse_loss(logits, labels)
#         metric = self.metric(logits, labels)
#         return loss, metric

#     def training_step(self, batch):
#         logits, labels = self.predict(batch)
#         loss, metric = self.cal_loss(logits, labels)
#         self.log('train_loss', loss, prog_bar=True, on_epoch=True)
#         self.log('train_mae', metric, prog_bar=True, on_epoch=True)
#         return loss

#     def validation_step(self, batch):
#         logits, labels = self.predict(batch, mc=True)
#         # Clamp predictions to valid range if needed
#         logits_clamped = torch.clamp(logits, min=0, max=46)
#         loss, metric = self.cal_loss(logits_clamped, labels)
#         self.log('val_loss', loss, prog_bar=True, on_epoch=True)
#         self.log('val_mae', metric, prog_bar=True, on_epoch=True)
#         return loss

#     def predict_step(self, batch):
#         logits, labels = self.predict(batch, mc=True)
#         logits_clamped = torch.clamp(logits, min=0, max=46)
#         return logits_clamped, labels

#     def configure_optimizers(self):
#         # Use different learning rates for backbone and head
#         backbone_params = []
#         head_params = []
        
#         for i, layer in enumerate(self.model):
#             if i < 10:  # Backbone layers
#                 backbone_params.extend([p for p in layer.parameters() if p.requires_grad])
#             else:  # Head layer
#                 head_params.extend([p for p in layer.parameters() if p.requires_grad])
        
#         optimizer = torch.optim.Adam([
#             {'params': backbone_params, 'lr': self.learning_rate * 0.1},  # Lower LR for backbone
#             {'params': head_params, 'lr': self.learning_rate}  # Higher LR for head
#         ])
        
#         # Optional: Add scheduler
#         scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
#             optimizer, mode='min', factor=0.5, patience=5
#         )
        
#         return {
#             "optimizer": optimizer,
#             "lr_scheduler": {
#                 "scheduler": scheduler,
#                 "monitor": "val_loss",
#             },
#         }

In [ ]:
class LiverFeatureExtract(L.LightningModule):
    def __init__(self, model, mc_iteration, learning_rate=1e-4, number_of_freeze_layers=0):
        super(LiverFeatureExtract, self).__init__()
        self.model = model
        self.metric = MeanAbsoluteError()
        self.mc_iteration = mc_iteration
        self.learning_rate = learning_rate

        self.save_hyperparameters(ignore=['model'])
    
        self.freeze_backbone_layers(number_of_freeze_layers)

    def freeze_backbone_layers(self, number_of_freeze_layers):
        """
        Freeze backbone layers, keeping only the last N layers trainable
        """
        # First, freeze all parameters
        for param in self.model.parameters():
            param.requires_grad = False
        
        # Get all children modules
        children = list(self.model.children())
        
        if number_of_freeze_layers == 0:
            # Unfreeze only the last layer (classification/regression head)
            if hasattr(self.model, 'classify'):
                for param in self.model.classify.parameters():
                    param.requires_grad = True
            elif hasattr(self.model, 'head'):
                for param in self.model.head.parameters():
                    param.requires_grad = True
            else:
                # Unfreeze last Sequential block
                for param in children[-1].parameters():
                    param.requires_grad = True
        else:
            # Unfreeze last N layers
            for layer in children[-number_of_freeze_layers:]:
                for param in layer.parameters():
                    param.requires_grad = True
        
        # Print trainable parameters info
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

    def forward(self, x):
        outputs = self.model(x)
        return outputs

    def predict(self, batch, mc=False):
        images, labels = batch

        if mc:
            # Enable dropout for MC sampling
            self.train()  # Set to training mode to enable dropout
            predictions = []
            
            with torch.no_grad():
                for _ in range(self.mc_iteration):
                    pred = self(images)
                    predictions.append(pred)
            
            # Stack predictions and compute mean
            logits = torch.stack(predictions).mean(dim=0).squeeze()
            self.eval()  # Set back to eval mode
        else:
            logits = self(images).squeeze()
        
        return logits, labels

    def cal_loss(self, logits, labels):
        loss = F.mse_loss(logits, labels)
        metric = self.metric( logits,labels)
        return loss, metric

    def training_step(self, batch):
        logits, labels = self.predict(batch)
        loss, metric = self.cal_loss(logits, labels)
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        loss, metric = self.cal_loss(logits_clamped, labels)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def predict_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        return logits_clamped, labels

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

In [ ]:
# Stage 1: Train only head (15 epochs)
model = LiverFeatureExtract(my_model, mc_iteration=10, number_of_freeze_layers=0)

trainer = L.Trainer(
    # accelerator="gpu",
    # devices=3,
    # strategy="ddp_notebook",  # for multi-GPU
    max_epochs=15,
    log_every_n_steps=10
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=eval_loader
)

In [ ]:
# Stage 2: Train 2 layer (20 epochs)
model.freeze_backbone_layers(number_of_freeze_layers=10)

trainer = L.Trainer(
    # accelerator="gpu",
    # devices=3,
    # strategy="ddp_notebook",  # for multi-GPU
    max_epochs=100,
    log_every_n_steps=10
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=eval_loader
)

In [ ]:
# Stage 3: Train 3 layer (20 epochs)
model.freeze_backbone_layers(number_of_freeze_layers=3)

trainer = L.Trainer(
    # accelerator="gpu",
    # devices=3,
    # strategy="ddp_notebook",  # for multi-GPU
    max_epochs=100,
    log_every_n_steps=10
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=eval_loader
)

In [ ]:
print("Final Train Loss:", trainer.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer.callback_metrics.get("val_loss"))

#### inference

In [ ]:
class CustomDataSetTest(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]  # ✅ include label

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.float32)
    
test_dataset = CustomDataSetTest(test_path, image_dir, transform=transformation)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [ ]:
predictions = trainer.predict(model, test_loader)

In [ ]:
# Extract only prediction tensors (assuming first item in tuple)
pred_tensors = [batch[0] for batch in predictions]  # extract y_hat

# Concatenate into a single tensor
all_pred = torch.cat(pred_tensors, dim=0)

# Convert to NumPy
pred_array = all_pred.detach().cpu().numpy()

# Check the number of predictions
print(len(pred_array))


In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv")
test_df["TE(kPa)"] = pred_array
test_df.head()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=test_df, x="TE(kPa)", bins=30, kde=False, color='skyblue')

plt.title("CNN Answer Distribution")
plt.xlabel("TE(kPa)")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
test_df.to_csv("submissionCon2D.csv",index = False)

# Vector from YOLO

### YOLO built

In [2]:
YOLO_model = YOLO("yolo11x-cls.yaml")
my_model = YOLO_model.model.model

YOLO11x-cls summary: 176 layers, 29,637,064 parameters, 29,637,064 gradients, 112.0 GFLOPs


In [3]:
train_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv"
# train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/500038-Moji/Segment_Liver"

In [4]:
train_df = pd.read_csv(train_path)
print(train_df)

      subject                image_name                    view  \
0           1  5c7e1166e138234adf000b5a             Intercostal   
1           2  5c809dc9e138233fae00064d             Intercostal   
2           3  5cedf3f5e13823798c000355  Subcostal_hepatic_vein   
3           3  5cedf408e13823798c00035e             Intercostal   
4           3  5cedf40ce13823798c000364                Liver/RK   
...       ...                       ...                     ...   
1767      686  64c1edefe13823510a000875             Intercostal   
1768      686  64c1ee77e1382371b300000c  Subcostal_hepatic_vein   
1769      687  64c472fce13823221100021a  Subcostal_hepatic_vein   
1770      687  64c4731fe138232211000226             Intercostal   
1771      687  64c4738ee13823221100023a                Liver/RK   

     SWE fibrosis stage  TE(kPa) TE result  
0                     -      3.4        F0  
1                     -      8.5        F2  
2                    F2      9.4        F3  
3              

In [5]:
# # Separate the F0 class and the rest
# f0 = train_df[train_df['TE result'] == 'F0']
# other_classes = train_df[train_df['TE result'] != 'F0']

# # Downsample F0 to 400 samples (random_state ensures reproducibility)
# f0_downsampled = f0.sample(n=400, random_state=42)

# # Combine back
# train_df = pd.concat([f0_downsampled, other_classes], ignore_index=True)

# # Optional: shuffle the final dataframe
# train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [6]:
class CustomDataSet(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)
        label = self.df["TE(kPa)"][index]

        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label,dtype = torch.float32)  # Ensure label is a tensor

In [7]:
BATCH_SIZE = 32
IMAGE_SIZE = 224
MEAN = [0.5,0.5,0.5]
STD = [0.5,0.5,0.5]

transformation = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])
dataset = CustomDataSet(train_df, image_dir, transform=transformation)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

eval_loader = DataLoader(
    dataset=eval_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [8]:
# my_model[10] = nn.Sequential(
#     nn.Conv2d(768, 256, kernel_size=1),
#     nn.ReLU(),
#     nn.AdaptiveAvgPool2d(1),
#     nn.Flatten(),
#     nn.Dropout(p=0.5),
#     nn.Linear(256, 1)
# )

my_model[10] = nn.Sequential(
    nn.Conv2d(768, 1280, kernel_size=1),
    nn.BatchNorm2d(1280),
    nn.ReLU(),
    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(),
    nn.Dropout(p=0.5),
    nn.Linear(1280, 1)
)

In [9]:
class LiverFeatureExtract(L.LightningModule):
    def __init__(self, model, mc_iteration):
        super(LiverFeatureExtract, self).__init__()
        self.model = model
        self.metric = MeanAbsoluteError()
        self.dropout = nn.Dropout()
        self.mc_iteration = mc_iteration

        for param in self.model.parameters():
            param.requires_grad = True

    def forward(self, x):
        outputs = self.model(x)
        return outputs

    def predict(self, batch, mc=False):
    # Safely handle batch whether it's (images, labels) or just images
        if isinstance(batch, (list, tuple)) and len(batch) == 2:
            images, labels = batch
        else:
            images = batch
            labels = None

        if mc:
            pred = [self(images).unsqueeze(0) for _ in range(self.mc_iteration)]
            logits = torch.vstack(pred).mean(dim=0).squeeze()
            return logits, labels

        logits = self(images).squeeze()
        return logits, labels

    def cal_loss(self, logits, labels):
        loss = F.mse_loss(logits, labels)
        metric = self.metric( logits,labels)
        return loss, metric

    def training_step(self, batch):
        logits, labels = self.predict(batch)
        loss, metric = self.cal_loss(logits, labels)
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        loss, metric = self.cal_loss(logits_clamped, labels)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_mae', metric, prog_bar=True, on_epoch=True)
        return loss

    def predict_step(self, batch):
        logits, labels = self.predict(batch, mc=True)
        logits_clamped = torch.clamp(logits, min=0, max=46)
        return logits_clamped, labels

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-4)

In [10]:
model = LiverFeatureExtract(my_model, 10)

In [11]:
trainer = L.Trainer(
    max_epochs=70,
    log_every_n_steps=10
)

trainer.fit(
    model=model,
    train_dataloaders=train_loader,
    val_dataloaders=eval_loader
)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/lustrefs/disk/project/ai901504-ai0004/501641_Big/week5/env_image/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA A100-SXM4-40GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_pre

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=70` reached.


In [12]:
print("Final Train Loss:", trainer.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer.callback_metrics.get("val_loss"))

Final Train Loss: tensor(0.7353)
Final Validation Loss: tensor(13.6385)


### ViT Model

In [13]:
ViT = ViTModel.from_pretrained('/project/ai901504-ai0004/VisionModels/project/ai901504-ai0004/VisionModels/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50')
ViT

Some weights of ViTModel were not initialized from the model checkpoint at /project/ai901504-ai0004/VisionModels/project/ai901504-ai0004/VisionModels/models--google--vit-large-patch16-224/snapshots/9e2727f4250d3973839eecfa5c4b42e41b709a50 and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


ViTModel(
  (embeddings): ViTEmbeddings(
    (patch_embeddings): ViTPatchEmbeddings(
      (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): ViTEncoder(
    (layer): ModuleList(
      (0-23): 24 x ViTLayer(
        (attention): ViTAttention(
          (attention): ViTSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (output): ViTSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
        )
        (intermediate): ViTIntermediate(
          (dense): Linear(in_features=1024, out_features=4096, bias=True)
          (intermediate_act_fn): GELUActivation()
        )
        (output): ViTOutput(
  

In [14]:
class LiverFibrosisModel(L.LightningModule):
    def __init__(self, model,num_classes=1):
        super(LiverFibrosisModel, self).__init__()

        # Load pretrained ViT base model
        self.vit = model
        # Freeze the ViT encoder
        for param in self.vit.parameters():
            param.requires_grad = True

        for param in self.vit.encoder.layer[-1:].parameters():
            param.requires_grad = True

        self.regressor = nn.Linear(self.vit.config.hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        outputs = self.vit(pixel_values=x)
        cls_token = outputs.last_hidden_state[:, 0]  # CLS token
        regression_output = self.regressor(cls_token)
        return self.relu(regression_output)

    def training_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        images, labels = batch
        preds = self(images).squeeze()
        val_loss = F.mse_loss(preds, labels.float().squeeze())
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True)
        return val_loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-5)


In [15]:
vit_model = LiverFibrosisModel(ViT)
vit_model

LiverFibrosisModel(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-23): 24 x ViTLayer(
          (attention): ViTAttention(
            (attention): ViTSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_features=1024, out_features=4096, bias=True)
            (inter

In [16]:
trainer_vit = L.Trainer(max_epochs=30,
                    log_every_n_steps = 10)

trainer_vit.fit(model=vit_model,
            train_dataloaders=train_loader,
            val_dataloaders = eval_loader
            )

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type     | Params | Mode 
-----------------------------------------------
0 | vit       | ViTModel | 304 M  | eval 
1 | regressor | Linear   | 1.0 K  | train
2 | relu      | ReLU     | 0      | train
-----------------------------------------------
304 M     Trainable params
0         Non-trainable params
304 M     Total params
1,217.409 Total estimated model params size (MB)
2         Modules in train mode
419       Modules in eval mode
SLURM auto-requeueing enabled. Setting signal handlers.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [17]:
print("Final Train Loss:", trainer_vit.callback_metrics.get("train_loss"))
print("Final Validation Loss:", trainer_vit.callback_metrics.get("val_loss"))

Final Train Loss: tensor(0.0184)
Final Validation Loss: tensor(13.1586)


### Prepare dataset

In [18]:
# train_path = "/project/ai901504-ai0004/501641_Big/week5/train_augment.csv"
train_path = '/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/train.csv'
test_path = "/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv"
image_dir = "/project/ai901504-ai0004/501641_Big/week5/dataset_2k"

In [19]:
class CustomDataSetTest(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, is_test=False):
        self.df = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        image_path = os.path.join(self.image_dir, self.df.image_name[index] + ".png")
        image = Image.open(image_path)


        if self.transform:
            image = self.transform(image)
        return image
    
train_df = pd.read_csv(train_path)
train_dataset = CustomDataSetTest(train_df, image_dir, transform=transformation)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_df = pd.read_csv(test_path)
test_dataset = CustomDataSetTest(test_df, image_dir, transform=transformation)
test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [20]:
predictions_train = trainer.predict(model, train_loader)
predictions_test = trainer.predict(model, test_loader)

predictions_train_vit = trainer_vit.predict(vit_model, train_loader)
predictions_test_vit = trainer_vit.predict(vit_model, test_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
SLURM auto-requeueing enabled. Setting signal handlers.


Predicting: |          | 0/? [00:00<?, ?it/s]

In [21]:
# Extract only prediction tensors (assuming first item in tuple)
pred_tensors = [pred[0] for pred in predictions_train]
all_pred = torch.cat(pred_tensors, dim=0)
pred_array_train = all_pred.detach().cpu().numpy()

print(len(pred_array_train))
  
pred_tensors = [pred[0] for pred in predictions_test]
all_pred = torch.cat(pred_tensors, dim=0)
pred_array_test = all_pred.detach().cpu().numpy()

print(len(pred_array_test))

# Extract only prediction tensors (assuming first item in tuple) 
all_pred = torch.cat(predictions_train_vit, dim=0)
pred_array_train_vit = all_pred.detach().cpu().numpy()

print(len(pred_array_train_vit))
  
all_pred = torch.cat(predictions_test_vit, dim=0)
pred_array_test_vit = all_pred.detach().cpu().numpy()

print(len(pred_array_test_vit))


1772
433
1772
433


In [22]:
train_df = pd.read_csv("/project/ai901504-ai0004/501641_Big/week5/regressor_train.csv")
test_df = pd.read_csv("/project/ai901504-ai0004/501641_Big/week5/regressor_test.csv")
train_df["TE(kPa)_from_regressor"] = pred_array_train
test_df["TE(kPa)_from_regressor"] = pred_array_test
train_df["TE(kPa)_from_regressor_vit"] = pred_array_train_vit
test_df["TE(kPa)_from_regressor_vit"] = pred_array_test_vit
train_df.head()
test_df.head()

,image_name,view,SWE fibrosis stage,TE(kPa),TE(kPa)_mean,TE(kPa)_median,TE(kPa)_max,TE(kPa)_min,TE(kPa)_std,TE_result_pct_F0,TE_result_pct_F1,TE_result_pct_F2,TE_result_pct_F3,TE_result_pct_F4,TE(kPa)_from_regressor,TE(kPa)_from_regressor_vit
0,5c7de4d6e138234adf00044d,Intercostal,-,8.8,8.495652,6.7,16.6,3.3,4.436112,0.369565,0.195652,0.108696,0.043478,0.282609,5.956312,5.547014
1,5d118de2e138237034000368,Liver/RK,F0-1,4.3,5.043525,4.6,46.0,2.4,2.498163,0.806102,0.120000,0.033220,0.021695,0.018983,5.895594,5.550655
2,5d1ae0b9e13823585300002f,Intercostal,F0-1,7.5,5.043525,4.6,46.0,2.4,2.498163,0.806102,0.120000,0.033220,0.021695,0.018983,3.881721,5.928147
3,5d1ae0c2e138236ca60005b9,Liver/RK,F0-1,NaN,5.043525,4.6,46.0,2.4,2.498163,0.806102,0.120000,0.033220,0.021695,0.018983,4.756416,6.118620
4,5d1ae0c4e138236ca60005bd,Subcostal_hepatic_vein,F0-1,NaN,5.043525,4.6,46.0,2.4,2.498163,0.806102,0.120000,0.033220,0.021695,0.018983,3.515515,6.388115


### Autogloun

In [23]:
print(train_df.columns)
print(test_df.columns)

Index(['subject', 'image_name', 'view', 'SWE fibrosis stage', 'TE(kPa)',
       'TE result', 'TE(kPa)_mean', 'TE(kPa)_median', 'TE(kPa)_max',
       'TE(kPa)_min', 'TE(kPa)_std', 'TE_result_pct_F0', 'TE_result_pct_F1',
       'TE_result_pct_F2', 'TE_result_pct_F3', 'TE_result_pct_F4',
       'TE(kPa)_from_regressor', 'TE(kPa)_from_regressor_vit'],
      dtype='object')
Index(['image_name', 'view', 'SWE fibrosis stage', 'TE(kPa)', 'TE(kPa)_mean',
       'TE(kPa)_median', 'TE(kPa)_max', 'TE(kPa)_min', 'TE(kPa)_std',
       'TE_result_pct_F0', 'TE_result_pct_F1', 'TE_result_pct_F2',
       'TE_result_pct_F3', 'TE_result_pct_F4', 'TE(kPa)_from_regressor',
       'TE(kPa)_from_regressor_vit'],
      dtype='object')


In [24]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [25]:
label = 'TE(kPa)'

In [26]:
train_df_auto = train_df.drop(columns=['subject', 'TE result'])

In [ ]:
predictor = TabularPredictor(label=label, eval_metric='root_mean_squared_error').fit(train_df_auto, presets='high_quality',
    ag_args_fit={'num_gpus': 1}, time_limit=1500)

No path specified. Models will be saved in: "AutogluonModels/ag-20250606_002853"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.3.1
Python Version:     3.10.18
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Wed Aug 30 20:32:41 UTC 2023 (dcbc95a)
CPU Count:          64
Memory Avail:       471.48 GB / 502.73 GB (93.8%)
Disk Space Avail:   7869.46 GB / 10240.00 GB (76.9%)
Presets specified: ['high_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid this risk by setting `save_bag_folds=True`.
DyStack is en

[1000]	valid_set's rmse: 2.01028


	Training S1F2 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F3 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F4 with GPU, note that this may negatively impact model quality compared to CPU training.


[1000]	valid_set's rmse: 1.84569


	Training S1F5 with GPU, note that this may negatively impact model quality compared to CPU training.


...

	Training S1F4 with GPU, note that this may negatively impact model quality compared to CPU training.


[1000]	valid_set's rmse: 2.22637


	Training S1F5 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F6 with GPU, note that this may negatively impact model quality compared to CPU training.


[1000]	valid_set's rmse: 2.95138
[2000]	valid_set's rmse: 2.88715
[3000]	valid_set's rmse: 2.87423
[4000]	valid_set's rmse: 2.86476


	Training S1F7 with GPU, note that this may negatively impact model quality compared to CPU training.
	Training S1F8 with GPU, note that this may negatively impact model quality compared to CPU training.
	-2.4464	 = Validation score   (-root_mean_squared_error)
	38.51s	 = Training   runtime
	0.03s	 = Validation runtime
Fitting model: NeuralNetFastAI_r145_BAG_L1 ... Training model for up to 55.40s of the 420.94s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
No improvement since epoch 7: early stopping
No improvement since epoch 1: early stopping
	-2.0997	 = Validation score   (-root_mean_squared_error)
	18.94s	 = Training   runtime
	0.06s	 = Validation runtime
Fitting model: XGBoost_r89_BAG_L1 ... Training model for up to 36.36s of the 401.90s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
/lustrefs/disk/project/ai901504-ai0004/501641_Big/week5/env_image/lib/python3.1

In [ ]:
y_pred = predictor.predict(test_df.drop(columns=[label]))

In [ ]:
y_pred

In [ ]:
test_df = pd.read_csv("/project/ai901504-ai0004/kaggle_competition/week5/liver-fibrosis-severity-prediction/test_submission.csv")
test_df["TE(kPa)"] = y_pred
print(test_df.head())

In [ ]:
# test_df["TE(kPa)"] -= 1

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=test_df, x="TE(kPa)", bins=30, kde=False, color='skyblue')

plt.title("CNN Answer Distribution")
plt.xlabel("TE(kPa)")
plt.ylabel("Count")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
test_df.to_csv("submissionYOLO.csv", index=False)